## Step 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings, time, pickle
warnings.filterwarnings('ignore')

from skrebate import ReliefF
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.impute          import SimpleImputer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics         import accuracy_score, f1_score
from sklearn.svm             import LinearSVC
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.neural_network  import MLPClassifier


## Step 2: Config

In [ ]:
DATASETS = {
    'Dataset9' : {'train': 'Datasets/9/train.csv',  'test': 'Datasets/9/test.csv'},
    'Dataset10': {'train': 'Datasets/10/train.csv', 'test': 'Datasets/10/test.csv'},
    'Dataset11': {'train': 'Datasets/11/train.csv', 'test': 'Datasets/11/test.csv'},
    'Dataset12': {'train': 'Datasets/12/train.csv', 'test': 'Datasets/12/test.csv'},
    'Dataset13': {'train': 'Datasets/13/train.csv', 'test': 'Datasets/13/test.csv'},
    'Dataset14': {'train': 'Datasets/14/train.csv', 'test': 'Datasets/14/test.csv'},
    'Dataset15': {'train': 'Datasets/15/train.csv', 'test': 'Datasets/15/test.csv'},
    'Dataset16': {'train': 'Datasets/16/train.csv', 'test': 'Datasets/16/test.csv'},
}

LABEL_COL           = 'Label'
N_FOLDS             = 10
INNER_FOLDS         = 3
RANDOM_STATE        = 42
K_CANDIDATES        = [50, 100, 150]
RELIEFF_N_NEIGHBORS = 10
CHECKPOINT_FILE     = 'results_checkpoint1.pkl'
EXCEL_FILE          = 'formatted_results.xlsx'


## Step 3: Classifiers

In [ ]:
CLASSIFIERS = {
    'SVM': (
        LinearSVC(random_state=RANDOM_STATE, max_iter=2000, dual='auto'),
        {'classifier__C': [1, 10]}
    ),
    'kNN': (
        KNeighborsClassifier(),
        {
            'classifier__n_neighbors': [3, 5, 11, 21],
            'classifier__weights':     ['uniform', 'distance'],
        }
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {
            'classifier__max_depth':        [5, 10, 20, None],
            'classifier__min_samples_split': [2, 5, 10],
        }
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {
            'classifier__n_estimators': [50, 100, 200],
            'classifier__max_depth':    [5, 10, None],
        }
    ),
    'MLP': (
        MLPClassifier(max_iter=500, random_state=RANDOM_STATE, early_stopping=True),
        {
            'classifier__hidden_layer_sizes': [(64,), (128,)],
            'classifier__alpha':              [1e-4, 1e-3],
        }
    ),
}


## Step 4: Helper Functions

In [ ]:
def load_dataset(path):
    df = pd.read_csv(path)
    X  = df.drop(columns=[LABEL_COL]).values.astype(np.float64)
    y  = LabelEncoder().fit_transform(df[LABEL_COL].values)
    print(f'  {X.shape[0]} samples | {X.shape[1]} features | {len(np.unique(y))} classes')
    return X, y


def apply_relieff(X_train, y_train, k, max_samples=500):
    rng = np.random.RandomState(RANDOM_STATE)
    if X_train.shape[0] > max_samples:
        idx      = rng.choice(X_train.shape[0], max_samples, replace=False)
        X_s, y_s = X_train[idx], y_train[idx]
    else:
        X_s, y_s = X_train, y_train
    X_s = SimpleImputer(strategy='mean').fit_transform(X_s)
    sel = ReliefF(n_features_to_select=k, n_neighbors=RELIEFF_N_NEIGHBORS)
    sel.fit(X_s, y_s)
    importances = sel.feature_importances_
    top_idx     = np.argsort(importances)[::-1][:k]
    return X_train[:, top_idx], importances, top_idx


def build_pipeline(clf):
    return Pipeline([
        ('imputer',    SimpleImputer(strategy='mean')),
        ('scaler',     StandardScaler()),
        ('classifier', clf),
    ])


def evaluate_classifier(clf, param_grid, X_tr, y_tr, X_te, y_te):
    gs = GridSearchCV(
        build_pipeline(clf), param_grid,
        cv=StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE),
        scoring='f1_macro', n_jobs=-1, refit=True)
    gs.fit(X_tr, y_tr)
    y_pred = gs.predict(X_te)
    return (accuracy_score(y_te, y_pred),
            f1_score(y_te, y_pred, average='macro', zero_division=0),
            gs.best_params_)


## Step 5: Main Experiment Loop (CV on train.csv)

In [ ]:
def training_models(datasets, classifiers):
    all_results    = {}
    relieff_scores = {}
    best_k_per_ds  = {}   # store best k per dataset for final test evaluation
    outer_cv       = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    total_start    = time.time()

    for ds_name, paths in datasets.items():
        print(f"\n{'='*60}\n  Dataset: {ds_name}\n{'='*60}")
        X, y = load_dataset(paths['train'])

        all_results[ds_name] = {
            'baseline': {c: {'acc': [], 'f1': [], 'params': []} for c in classifiers},
            'relieff':  {c: {'acc': [], 'f1': [], 'params': []} for c in classifiers},
        }
        fold_feature_scores = []
        fold_best_ks        = []

        for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
            print(f"\n  ── Fold {fold_idx+1}/{N_FOLDS} ──")
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            # Phase 1: Baseline
            print("  [Baseline]")
            for clf_name, (clf, grid) in classifiers.items():
                t0 = time.time()
                acc, f1, params = evaluate_classifier(clf, grid, X_tr, y_tr, X_te, y_te)
                all_results[ds_name]['baseline'][clf_name]['acc'].append(acc)
                all_results[ds_name]['baseline'][clf_name]['f1'].append(f1)
                all_results[ds_name]['baseline'][clf_name]['params'].append(params)
                print(f"    {clf_name:15s}  acc={acc:.4f}  f1={f1:.4f}  ({time.time()-t0:.1f}s)")

            # Phase 2: ReliefF — rank features
            print("  [ReliefF] ranking features …", end=" ", flush=True)
            t0 = time.time()
            _, importances, _ = apply_relieff(X_tr, y_tr, int(max(K_CANDIDATES)))
            print(f"{time.time()-t0:.1f}s")

            # Pick best k via inner CV with RF proxy
            best_k, best_score = K_CANDIDATES[0], -np.inf
            inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
            for k in K_CANDIDATES:
                if k > X_tr.shape[1]:
                    continue
                Xtr_k  = X_tr[:, np.argsort(importances)[::-1][:k]]
                scores = []
                for itr, ite in inner.split(Xtr_k, y_tr):
                    rf = RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1)
                    rf.fit(Xtr_k[itr], y_tr[itr])
                    scores.append(f1_score(y_tr[ite], rf.predict(Xtr_k[ite]),
                                           average='macro', zero_division=0))
                if np.mean(scores) > best_score:
                    best_score, best_k = np.mean(scores), k

            top_final = np.argsort(importances)[::-1][:best_k]
            X_tr_r, X_te_r = X_tr[:, top_final], X_te[:, top_final]
            fold_feature_scores.append(importances)
            fold_best_ks.append(best_k)
            print(f"  [ReliefF] best_k={best_k}")

            # Phase 2: ReliefF classifiers
            for clf_name, (clf, grid) in classifiers.items():
                t0 = time.time()
                acc, f1, params = evaluate_classifier(clf, grid, X_tr_r, y_tr, X_te_r, y_te)
                all_results[ds_name]['relieff'][clf_name]['acc'].append(acc)
                all_results[ds_name]['relieff'][clf_name]['f1'].append(f1)
                all_results[ds_name]['relieff'][clf_name]['params'].append(params)
                print(f"    {clf_name:15s}  acc={acc:.4f}  f1={f1:.4f}  ({time.time()-t0:.1f}s)")

        relieff_scores[ds_name] = np.mean(fold_feature_scores, axis=0)
        best_k_per_ds[ds_name]  = int(np.median(fold_best_ks))  # most common k across folds

        with open(CHECKPOINT_FILE, 'wb') as f:
            pickle.dump({'all_results': all_results,
                         'relieff_scores': relieff_scores,
                         'best_k_per_ds': best_k_per_ds}, f)
        print(f"\n  ✅  Checkpoint saved after {ds_name}")

    print(f"\n{'='*60}\n  Done in {(time.time()-total_start)/60:.1f} min\n{'='*60}")
    return all_results, relieff_scores, best_k_per_ds


all_results, relieff_scores, best_k_per_ds = training_models(DATASETS, CLASSIFIERS)


## Step 6: Final Test Evaluation (test.csv)

In [ ]:
# Load checkpoint if re-running this cell independently
if 'best_k_per_ds' not in dir():
    with open(CHECKPOINT_FILE, 'rb') as f:
        ck = pickle.load(f)
    all_results    = ck['all_results']
    relieff_scores = ck['relieff_scores']
    best_k_per_ds  = ck['best_k_per_ds']

final_test_results = {}

for ds_name, paths in DATASETS.items():
    print(f'\n=== Final Test: {ds_name} ===')
    X_train, y_train = load_dataset(paths['train'])
    X_test,  y_test  = load_dataset(paths['test'])

    final_test_results[ds_name] = {'baseline': {}, 'relieff': {}}

    # Baseline
    for clf_name, (clf, grid) in CLASSIFIERS.items():
        acc, f1, params = evaluate_classifier(clf, grid, X_train, y_train, X_test, y_test)
        final_test_results[ds_name]['baseline'][clf_name] = {'acc': acc, 'f1': f1, 'params': params}
        print(f'  [Baseline] {clf_name:15s}  acc={acc:.4f}  f1={f1:.4f}')

    # ReliefF — use median best_k from CV folds
    best_k  = best_k_per_ds.get(ds_name, K_CANDIDATES[0])
    _, importances, _ = apply_relieff(X_train, y_train, int(max(K_CANDIDATES)))
    top_idx = np.argsort(importances)[::-1][:best_k]
    X_train_r, X_test_r = X_train[:, top_idx], X_test[:, top_idx]

    for clf_name, (clf, grid) in CLASSIFIERS.items():
        acc, f1, params = evaluate_classifier(clf, grid, X_train_r, y_train, X_test_r, y_test)
        final_test_results[ds_name]['relieff'][clf_name] = {'acc': acc, 'f1': f1, 'params': params}
        print(f'  [ReliefF]  {clf_name:15s}  acc={acc:.4f}  f1={f1:.4f}')

with open('final_test_results.pkl', 'wb') as f:
    pickle.dump(final_test_results, f)
print('\n✅  final_test_results.pkl saved.')


## Step 7: Export Excel

In [ ]:
def export_excel(all_results, excel_file=EXCEL_FILE):
    model_map = {'SVM': 'SVM', 'kNN': 'KNN',
                 'Decision Tree': 'DT', 'Random Forest': 'RF', 'MLP': 'MLP'}

    def build_sheet(phase):
        all_rows = []
        for i, ds in enumerate(sorted(all_results.keys()), 1):
            block = all_results[ds][phase]
            cols  = [f'{m}-{metric}' for m in model_map.values()
                     for metric in ['Accuracy', 'F1']]
            cols += [f'{m} Parameters' for m in model_map.values()]

            all_rows.append([f'Data {i}'] + [''] * len(cols))
            all_rows.append([''] + cols)

            fold_data = []
            for fold in range(len(next(iter(block.values()))['acc'])):
                row, num_row = [f'Fold {fold+1}'], []
                for model in model_map:
                    acc, f1 = block[model]['acc'][fold], block[model]['f1'][fold]
                    row += [round(acc, 4), round(f1, 4)]
                    num_row += [acc, f1]
                for model in model_map:
                    row.append(str(block[model]['params'][fold]))
                all_rows.append(row)
                fold_data.append(num_row)

            arr = np.array(fold_data)
            mean_std = [f'{m:.4f} ± {s:.4f}'
                        for m, s in zip(np.mean(arr, axis=0), np.std(arr, axis=0))]
            all_rows.append(['Mean ± Std'] + mean_std + [''] * len(model_map))
            all_rows.append([''] * (len(cols) + 1))
        return pd.DataFrame(all_rows)

    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        build_sheet('baseline').to_excel(writer, sheet_name='Baseline', index=False, header=False)
        build_sheet('relieff').to_excel(writer,  sheet_name='ReliefF',  index=False, header=False)
    print(f'✅  Excel saved → {excel_file}')

export_excel(all_results)
